# 04 - Live PnL Analysis

NAV curve and drawdown from the event log. Uses
`guardrail_lab.metrics.nav_series` for the NAV curve and
`guardrail_lab.drawdown` for the underwater series and worst
drawdown episodes. Charts are rendered as text sparklines so the
notebook runs with the base stack (no matplotlib required).

**Offline-safe:** an empty NAV history prints a hint.


In [ ]:
# --- Guardrail Alpha notebook bootstrap (offline-safe) ---
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    """Return the first ancestor that contains python-lab/guardrail_lab."""
    for candidate in [start, *start.parents]:
        if (candidate / "python-lab" / "guardrail_lab").is_dir():
            return candidate
    return start


# In a notebook __file__ is undefined, so anchor on the current working dir.
REPO_ROOT = _find_repo_root(Path.cwd())
LAB_PATH = REPO_ROOT / "python-lab"
if str(LAB_PATH) not in sys.path:
    sys.path.insert(0, str(LAB_PATH))


def _first_existing(*candidates: Path):
    """Return the first path that exists, else None."""
    for path in candidates:
        if path.exists():
            return path
    return None


DATA_DIR = REPO_ROOT / "data"
CONFIG_DIR = REPO_ROOT / "configs"

# Prefer a real run; fall back to the seeded demo artifacts if present.
DB_PATH = _first_existing(
    DATA_DIR / "guardrail_alpha.db",
    DATA_DIR / "demo_guardrail_alpha.db",
)
RUN_REPORT_PATH = _first_existing(
    DATA_DIR / "run_report.json",
    DATA_DIR / "demo_run_report.json",
)
EXPERIMENTS_DIR = DATA_DIR / "experiments"
BACKTESTS_DIR = DATA_DIR / "backtests"
ELIGIBLE_ASSETS_PATH = CONFIG_DIR / "eligible_assets.bsc.json"
ASSET_CATEGORIES_PATH = CONFIG_DIR / "asset_categories.json"

NO_DATA_HINT = (
    "No data found under data/. Run the agent (paper/live) or seed a demo run "
    "first, e.g.  python3 -m guardrail_lab.seed  (from python-lab/), then "
    "re-run this notebook. The notebook is offline-safe and will not raise."
)

print("repo root :", REPO_ROOT)
print("event log :", DB_PATH if DB_PATH else "(none yet)")
print("run report:", RUN_REPORT_PATH if RUN_REPORT_PATH else "(none yet)")


In [ ]:
def render_table(rows, columns, title=None):
    """Print an aligned text table. rows: list[dict]; columns: list[(key,label)]."""
    if title:
        print(title)
        print("=" * len(title))
    if not rows:
        print("(no rows)")
        return
    labels = [label for _, label in columns]
    keys = [key for key, _ in columns]
    cells = [[("" if r.get(k) is None else str(r.get(k))) for k in keys] for r in rows]
    widths = [
        max(len(labels[i]), *(len(row[i]) for row in cells)) for i in range(len(keys))
    ]
    header = "  ".join(labels[i].ljust(widths[i]) for i in range(len(keys)))
    print(header)
    print("  ".join("-" * widths[i] for i in range(len(keys))))
    for row in cells:
        print("  ".join(row[i].ljust(widths[i]) for i in range(len(keys))))


## NAV curve


In [ ]:
from guardrail_lab.db import load_events
from guardrail_lab.metrics import nav_series, max_drawdown, trade_count

events = load_events(str(DB_PATH)) if DB_PATH else []
curve = nav_series(events)

if not curve:
    print(NO_DATA_HINT)
else:
    navs = [nav for _, nav in curve]
    start, end = navs[0], navs[-1]
    ret_pct = (end - start) / start * 100.0 if start else 0.0
    print(f"NAV points     : {len(curve)}")
    print(f"Starting NAV   : {start:,.2f}")
    print(f"Latest NAV     : {end:,.2f}")
    print(f"Total return   : {ret_pct:+.3f}%")
    print(f"Peak NAV       : {max(navs):,.2f}")
    print(f"Trough NAV     : {min(navs):,.2f}")
    print(f"Max drawdown   : {max_drawdown(navs) * 100.0:.3f}%")
    print(f"Confirmed trades: {trade_count(events)}")


## NAV sparkline (text)


In [ ]:
def sparkline(values):
    """ASCII sparkline using block characters; safe for any terminal."""
    if not values:
        return "(empty)"
    blocks = "▁▂▃▄▅▆▇█"
    lo, hi = min(values), max(values)
    span = (hi - lo) or 1.0
    return "".join(blocks[min(7, int((v - lo) / span * 7))] for v in values)

if curve:
    navs = [nav for _, nav in curve]
    print("NAV:", sparkline(navs))
    render_table(
        [
            {"timestamp": ts, "nav_usd": f"{nav:,.2f}"}
            for ts, nav in curve
        ],
        [("timestamp", "TIMESTAMP"), ("nav_usd", "NAV_USD")],
        title="NAV series",
    )
else:
    print(NO_DATA_HINT)


## Drawdown analysis


In [ ]:
from guardrail_lab.drawdown import analyze_drawdown_from_events

if events and curve:
    report = analyze_drawdown_from_events(events, top_n=5)
    print(f"Max drawdown   : {report.max_drawdown_pct:.4f}%")
    print(f"Peak           : {report.peak_timestamp or 'n/a'}")
    print(f"Trough         : {report.trough_timestamp or 'n/a'}")
    dd_secs = report.max_drawdown_seconds
    rec_secs = report.max_recovery_seconds
    print(f"Drawdown dur.  : {dd_secs if dd_secs is not None else 'n/a'} s")
    print(f"Recovery dur.  : {rec_secs if rec_secs is not None else 'n/a (unrecovered)'} s\n")

    underwater = [p.drawdown_pct for p in report.points]
    print("Underwater:", sparkline([-x for x in underwater]) if underwater else "(empty)")
    print()

    if report.episodes:
        render_table(
            [
                {
                    "depth_pct": f"{ep.depth_pct:.4f}%",
                    "peak": ep.peak_timestamp,
                    "trough": ep.trough_timestamp,
                    "recovered": ep.recovered,
                }
                for ep in report.episodes
            ],
            [
                ("depth_pct", "DEPTH"),
                ("peak", "PEAK_TS"),
                ("trough", "TROUGH_TS"),
                ("recovered", "RECOVERED"),
            ],
            title="Worst drawdown episodes",
        )
    else:
        print("No drawdown episodes (NAV never declined below a prior peak).")
else:
    print(NO_DATA_HINT)


## Notes

* NAV is taken from `portfolio_reconciled` events; each one carries
  a high-precision `nav_usd`.
* The underwater sparkline is inverted (taller = deeper drawdown).
